# Phase 3: Real-time Simulation & Deployment
## 1. Import Libraries

In [10]:
import pandas as pd
import numpy as np
import joblib
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## 2. Load Models and Scaler

In [11]:
import joblib
import os

try:
    # Load the best pipeline (contains both scaling and model logic)
    model_files = [f for f in os.listdir('../models') if f.endswith('_best.pkl')]
    if not model_files:
        raise FileNotFoundError("No best model found in ../models. Run Notebook 02 first.")
    
    model_path = os.path.join('../models', model_files[0])
    model = joblib.load(model_path)
    le = joblib.load('../models/label_encoder.pkl')
    selected_features = joblib.load('../models/selected_features.pkl')
    
    print(f"SUCCESS: Loaded model from {model_path}")
    print(f"Features expected: {len(selected_features)}")
except Exception as e:
    print(f"ERROR: {e}")


SUCCESS: Loaded model from ../models\logistic_regression_best.pkl
Features expected: 18


## 3. Real-time Prediction Function

In [12]:
def predict_flow(flow_dict):
    """
    Predicts intrusion for a single flow with confidence score.
    Returns: (predicted_label, confidence_score)
    """
    import pandas as pd
    import numpy as np
    from datetime import datetime
    
    # 1. Prepare data
    df_flow = pd.DataFrame([flow_dict])
    
    # Ensure all selected features are present, fill missing with 0 and log warning
    missing_features = [f for f in selected_features if f not in df_flow.columns]
    if missing_features:
        # Only print for the first few to avoid spam
        # print(f"Warning: Missing features in flow: {missing_features}")
        for f in missing_features:
            df_flow[f] = 0.0
            
    X_input = df_flow[selected_features]
    
    # 2. Prediction with Confidence
    try:
        # Get label prediction
        pred_encoded = model.predict(X_input)[0]
        pred_label = le.inverse_transform([pred_encoded])[0]
        
        # Get confidence (probability)
        if hasattr(model, 'predict_proba'):
            probs = model.predict_proba(X_input)[0]
            confidence = np.max(probs)
        else:
            confidence = 1.0 # Fallback for models without proba
            
        # 3. Alerting Logic
        if pred_label != 'BENIGN':
            timestamp = datetime.now().isoformat()
            severity = "HIGH" if confidence > 0.8 else "MEDIUM"
            
            alert_msg = f"[{timestamp}] [{severity}] ALERT: {pred_label} detected (Confidence: {confidence:.2%})\n"
            
            # Print to console (Red)
            print(f"\033[91m{alert_msg.strip()}\033[0m")
            
            # Log to file
            with open('../alerts.log', 'a', encoding='utf-8') as f_log:
                f_log.write(alert_msg)
        else:
            # print(f"\033[92mNormal traffic (BENIGN)\033[0m")
            pass
            
        return pred_label, confidence
        
    except Exception as e:
        print(f"Prediction Error: {e}")
        return "ERROR", 0.0


## 4. Test the Simulation

In [15]:
import pandas as pd
import os
import json
import uuid
from datetime import datetime

try:
    result_dir = '../result'
    os.makedirs(result_dir, exist_ok=True)
    
    # 1. Load and Sample Data
    NUM_SAMPLES = 500
    df_all = pd.read_parquet('../datasets/cleaned_data.parquet')
    
    df_benign = df_all[df_all['Label'] == 'BENIGN'].sample(NUM_SAMPLES, random_state=42)
    df_attacks = df_all[df_all['Label'] != 'BENIGN'].sample(min(NUM_SAMPLES, len(df_all[df_all['Label'] != 'BENIGN'])), random_state=42)
    df_sample = pd.concat([df_benign, df_attacks]).sample(frac=1, random_state=42)
    
    print(f"Starting SIEM-ready simulation on {len(df_sample)} flows...\n")
    
    json_results = []
    correct_count = 0
    
    # 2. Simulation Loop
    for _, row in df_sample.iterrows():
        flow_data = row.drop('Label').to_dict()
        true_label = row['Label']
        
        # Predict
        pred_label, confidence = predict_flow(flow_data)
        
        if pred_label == true_label:
            correct_count += 1
            
        # 3. Create Enhanced JSON Output (SIEM Format)
        event = {
            "event_id": str(uuid.uuid4()),
            "timestamp": datetime.now().isoformat(),
            "sensor_id": "IDS-NODE-01",
            "destination_port": flow_data.get('Destination Port', 'Unknown'),
            "true_label": true_label,
            "predicted_label": pred_label,
            "confidence": round(float(confidence), 4),
            "severity": "HIGH" if pred_label != 'BENIGN' and confidence > 0.8 else ("MEDIUM" if pred_label != 'BENIGN' else "LOW"),
            "status": "SUCCESS" if pred_label == true_label else "FAILURE",
            "message": f"IDS Alert: {pred_label} detected on port {flow_data.get('Destination Port', 'Unknown')}" if pred_label != 'BENIGN' else "Normal traffic"
        }
        json_results.append(event)
        
    # 4. Save results
    output_path = os.path.join(result_dir, 'simulation_output.json')
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(json_results, f, indent=4)
        
    print("\n" + "-" * 50)
    print("SIMULATION COMPLETE.")
    print(f"- Total samples tested : {len(df_sample)}")
    print(f"- Total correct predictions : {correct_count}")
    print(f"- Total false predictions : {len(df_sample) - correct_count}")
    print(f"- Accuracy             : {(correct_count / len(df_sample)) * 100:.2f}%")
    print(f"\nResults saved to: {json_path}")

except Exception as e:
    print(f"Simulation Error: {e}")


Starting SIEM-ready simulation on 1000 flows...

[2026-05-04T15:05:19.811280] [MEDIUM] ALERT: DDoS detected (Confidence: 60.40%)
[2026-05-04T15:05:19.814552] [MEDIUM] ALERT: DDoS detected (Confidence: 72.62%)
[2026-05-04T15:05:19.819138] [HIGH] ALERT: DDoS detected (Confidence: 99.93%)
[2026-05-04T15:05:19.821144] [HIGH] ALERT: DoS Hulk detected (Confidence: 96.54%)
[2026-05-04T15:05:19.825059] [HIGH] ALERT: DDoS detected (Confidence: 99.83%)
[2026-05-04T15:05:19.827077] [HIGH] ALERT: DDoS detected (Confidence: 99.33%)
[2026-05-04T15:05:19.832108] [HIGH] ALERT: DoS Hulk detected (Confidence: 99.30%)
[2026-05-04T15:05:19.834196] [HIGH] ALERT: DoS Hulk detected (Confidence: 99.86%)
[2026-05-04T15:05:19.839027] [HIGH] ALERT: DDoS detected (Confidence: 99.91%)
[2026-05-04T15:05:19.843887] [HIGH] ALERT: DoS Hulk detected (Confidence: 99.48%)
[2026-05-04T15:05:19.848076] [MEDIUM] ALERT: DDoS detected (Confidence: 68.00%)
[2026-05-04T15:05:19.851855] [HIGH] ALERT: PortScan detected (Confidenc